In [ ]:
!nvidia-smi

Sun Nov 22 23:49:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
# Create a folder for the project
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Sanger_Project")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "outputs" / "glue").mkdir(
    parents=True, exist_ok=True
)

print("Project folder:", DRIVE_ROOT)

Project folder: /content/drive/MyDrive/Sanger_Project


In [29]:
!git clone https://github.com/pku-liang/Sanger.git /content/Sanger

Cloning into '/content/Sanger'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 70 (delta 19), reused 70 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 95.91 KiB | 13.70 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [30]:
%cd /content/Sanger

/content/Sanger


In [31]:
!ls
!ls scripts/
!ls configs/

bench_cpu_gpu.py    hardware		       quant_utils.py
bench_cusparse.cu   modeling_bart.py	       README.md
bench_gpu_qk_pv.py  modeling_bert.py	       requirements.txt
bench_sanger.py     modeling_gpt2.py	       run_cloth.py
bench_sanger_v2.py  modeling_sanger_spattn.py  run_glue.py
configs		    modeling_static_spattn.py  run_squad.py
data		    outputs		       scripts
eval_dense_on_cloth.sh	 eval_sparse_on_glue.sh   train_dense_on_squad.sh
eval_dense_on_glue.sh	 eval_sparse_on_squad.sh  train_sparse_on_cloth.sh
eval_dense_on_squad.sh	 train_dense_on_cloth.sh  train_sparse_on_glue.sh
eval_sparse_on_cloth.sh  train_dense_on_glue.sh   train_sparse_on_squad.sh
bart_base_sanger.json	   bert_base_sanger_2e-2.json  models
bert_base_bigbird.json	   bert_base_sanger_2e-3.json  sparse_attn
bert_base_longformer.json  gpt2_sanger_2e-3.json


In [32]:
import os

os.environ["PROJ_ROOT"] = "/content/Sanger"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

print(os.environ["PROJ_ROOT"])

/content/Sanger


In [18]:
!pip install -q uv

In [ ]:
!uv python install 3.8

Python 3.8 is already installed


In [12]:
!uv venv /content/sanger-env --python 3.8

Using CPython 3.8.20
Creating virtual environment at: /content/sanger-env
Activate with: source /content/sanger-env/bin/activate


In [19]:
!/content/sanger-env/bin/python --version

Python 3.8.20


In [34]:
!uv pip install \
  --python /content/sanger-env/bin/python \
  "numpy==1.21.6" \
  "pyarrow==3.0.0" \
  "pandas==1.3.5"

Using Python 3.8.20 environment at: /content/sanger-env
Checked 3 packages in 15ms


In [35]:
!/content/sanger-env/bin/python -m pip --version

pip 25.0.1 from /content/sanger-env/lib/python3.8/site-packages/pip (python 3.8)


In [36]:
!uv pip install \
  --python /content/sanger-env/bin/python \
  pip

Using Python 3.8.20 environment at: /content/sanger-env
Checked 1 package in 2ms


In [37]:
!/content/sanger-env/bin/python -m pip install \
  "torch==1.10.2+cu113" \
  -f https://download.pytorch.org/whl/cu113/torch_stable.html

Looking in links: https://download.pytorch.org/whl/cu113/torch_stable.html


In [38]:
%cd /content/Sanger

/content/Sanger


In [39]:
%%writefile /content/Sanger/constraints.txt
torch==1.10.2+cu113
numpy==1.21.6
pyarrow==3.0.0
pandas==1.3.5

Writing /content/Sanger/constraints.txt


In [40]:
!/content/sanger-env/bin/python -m pip install \
  -r requirements.txt \
  -c constraints.txt \
  -f https://download.pytorch.org/whl/cu113/torch_stable.html

Looking in links: https://download.pytorch.org/whl/cu113/torch_stable.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 154.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 159.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.1/785.1 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.2/555.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 166.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB

In [ ]:
# Connect Sanger directory to Google Drive
from pathlib import Path

local_output = Path("/content/Sanger/outputs/glue")
drive_output = Path(
    "/content/drive/MyDrive/Sanger_Project/outputs/glue"
)

drive_output.mkdir(parents=True, exist_ok=True)

if local_output.is_symlink():
    local_output.unlink()
elif local_output.exists():
    # Preserve any existing local output directory
    backup = Path("/content/Sanger/outputs/glue_before_link")
    if backup.exists():
        raise RuntimeError("Backup directory already exists.")
    local_output.rename(backup)

local_output.parent.mkdir(parents=True, exist_ok=True)
local_output.symlink_to(drive_output, target_is_directory=True)

print("Local path points to:", local_output.resolve())

Local path points to: /content/drive/MyDrive/Sanger_Project/outputs/glue


In [ ]:
%cd /content/Sanger

/content/Sanger


**Test 1: Software Prunning Implementation**

**Step 1: Training dense Bert on Glue datasets**




In [ ]:
!PATH=/content/sanger-env/bin:$PATH \
  bash scripts/train_dense_on_glue.sh \
  mrpc \
  bert-base-uncased \
  3 \
  2e-5 \
  32

Using the `WAND_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
11/23/2026 00:26:23 - WARNING - __main__ -   Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
11/23/2026 00:26:23 - INFO - __main__ -   Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_find_unused_parameters=None,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=True,
eval_accumulation_steps=None,
eval_steps=500,
evaluation_strategy=IntervalStrategy.NO,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
gradient_accumulation_steps=1,
greater_is_better=None,
group_by_length=False,
ignore_data_skip=False,
label_names=None,
la

**Evaluate the training result on dense Bert with GLUE data (use to evaluate trained data without traing data again)**

In [31]:
#!PATH=/content/sanger-env/bin:$PATH \
#  bash scripts/eval_dense_on_glue.sh \
#  mrpc \
#  bert-base-uncased

Using the `WAND_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
11/23/2026 00:55:24 - WARNING - __main__ -   Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
11/23/2026 00:55:24 - INFO - __main__ -   Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_find_unused_parameters=None,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_steps=500,
evaluation_strategy=IntervalStrategy.NO,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
gradient_accumulation_steps=1,
greater_is_better=None,
group_by_length=False,
ignore_data_skip=False,
label_names=None,
l

**Train spare Sanger-based BERT model**

In [5]:
from pathlib import Path
import os
import subprocess

repo = Path("/content/drive/MyDrive/Sanger")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/hatsu3/Sanger.git", str(repo)],
        check=True,
    )

os.chdir(repo)
os.environ["PROJ_ROOT"] = str(repo)
print("Training script found:", Path("scripts/train_sparse_on_glue.sh").is_file())

Training script found: True


In [41]:
%%bash
set -e

PY=/content/sanger-env/bin/python
REPO=/content/drive/MyDrive/Sanger

"$PY" -m pip install \
  -r "$REPO/requirements.txt" \
  -c /content/Sanger/constraints.txt \
  -f https://download.pytorch.org/whl/cu113/torch_stable.html

"$PY" -c "from datasets import load_dataset, load_metric; print('Sanger imports OK')"

Looking in links: https://download.pytorch.org/whl/cu113/torch_stable.html
Sanger imports OK


In [42]:
!PATH=/content/sanger-env/bin:$PATH \
  bash scripts/train_sparse_on_glue.sh \
  mrpc \
  bert-base-uncased \
  bert_base_sanger_2e-2.json \
  3 \
  2e-5 \
  32

Using the `WAND_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
11/23/2026 02:37:16 - WARNING - __main__ -   Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
11/23/2026 02:37:16 - INFO - __main__ -   Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_find_unused_parameters=None,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=True,
eval_accumulation_steps=None,
eval_steps=500,
evaluation_strategy=IntervalStrategy.NO,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
gradient_accumulation_steps=1,
greater_is_better=None,
group_by_length=False,
ignore_data_skip=False,
label_names=None,
la

In [ ]:
%%bash
set -e
export PROJ_ROOT=/content/Sanger
export PATH="/content/sanger-env/bin:$PATH"
cd "$PROJ_ROOT"

# Preserve any previous measurement before Sanger creates a new CSV.
if [ -f load_balance.csv ]; then
  mv load_balance.csv "load_balance_$(date +%Y%m%d_%H%M%S).csv"
fi

bash scripts/eval_sparse_on_glue.sh \
  mrpc bert-base-uncased bert_base_sanger_2e-2.json true

python - <<'PY'
import pandas as pd

kept = pd.read_csv("load_balance.csv")["overall-sparsity"].dropna()
if kept.empty:
    raise SystemExit("No attention samples were logged. Rerun this evaluation cell.")

print(f"Attention kept:   {kept.mean():.2%}")
print(f"Attention pruned: {1 - kept.mean():.2%}")
PY

In [44]:
import pandas as pd

log = pd.read_csv("/content/Sanger/load_balance.csv")
kept = log["overall-sparsity"].dropna()

if kept.empty:
    print("The CSV exists but contains no attention samples.")
else:
    print(f"Attention kept:   {kept.mean():.2%}")
    print(f"Attention pruned: {1 - kept.mean():.2%}")

Attention kept:   16.61%
Attention pruned: 83.39%


**Test 2: PE Utilization with and without packaging**

**Generate PE Utilization Result**

In [45]:
!rm -f /content/Sanger/load_balance.csv

In [46]:
!PATH=/content/sanger-env/bin:$PATH \
  bash scripts/eval_sparse_on_glue.sh \
  mrpc \
  bert-base-uncased \
  bert_base_sanger_2e-2.json \
  true

Using the `WAND_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
11/23/2026 03:18:24 - WARNING - __main__ -   Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
11/23/2026 03:18:24 - INFO - __main__ -   Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_find_unused_parameters=None,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_steps=500,
evaluation_strategy=IntervalStrategy.NO,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
gradient_accumulation_steps=1,
greater_is_better=None,
group_by_length=False,
ignore_data_skip=False,
label_names=None,
l

In [47]:
!head -n 5 load_balance.csv

50%-no-skip,50%-skip,25%-no-skip,25%-skip,overall-sparsity
0.124392,0.248784,0.241956,0.470988,0.186091
0.093579,0.187158,0.186124,0.370205,0.156944
0.173012,0.346024,0.314188,0.575435,0.223412
0.082037,0.164073,0.163664,0.326513,0.119404


**Run hardware performance estimator**

In [48]:
!/content/sanger-env/bin/python bench_sanger.py \
  --csv_file load_balance.csv \
  --seq_len 128

Average Sparsity: 0.166
Load Balance (50%-no-skip): 0.115
Sanger Latency (50%-no-skip): 0.363 ms
Load Balance (50%-skip): 0.230
Sanger Latency (50%-skip): 0.292 ms
Load Balance (25%-no-skip): 0.222
Sanger Latency (25%-no-skip): 0.295 ms
Load Balance (25%-skip): 0.430
Sanger Latency (25%-skip): 0.259 ms


In [49]:
import shutil

shutil.copy2(
    "/content/Sanger/load_balance.csv",
    "/content/drive/MyDrive/Sanger_Project/load_balance.csv"
)

print("Load-balance results saved!")

Load-balance results saved!


In [50]:
!find /content/drive/MyDrive/Sanger_Project \
  -name "pytorch_model.bin"

/content/drive/MyDrive/Sanger_Project/outputs/glue/bert-base-uncased_mrpc/pytorch_model.bin
